# 09 · Databricks workspace helpers

This notebook drives `dtrack_left.backends.databricks` against your real workspace: `workspace` (profile -> `WorkspaceClient`), `login`, `upload_workspace`, `compute_status/start/stop`, `warehouse_status/start/stop`, `run_sql`, `list_runs` and `run_detail`.

It mirrors `tests/unit/dtrack_left/backends/test_databricks.py`, which uses a recording stand-in for the SDK client. Here every read goes to the real workspace. Anything that changes the workspace (uploads, starting/stopping compute) only happens when `DRY_RUN = False`.

## How to approach

**Prerequisites**
- Kernel: the repo `.venv` with `uv sync --extra dbx` (databricks-sdk).
- A `~/.databrickscfg` profile (`DTRACK_DBX_PROFILE`, default `corp`); external-browser profiles open a login tab the first time.
- `DTRACK_DBX_WAREHOUSE_ID` for the SQL sections; `CLUSTER_ID` / `JOB_ID` are optional (the notebook lists candidates when they are empty).
- `DTRACK_PROD_CONFIG` / `DTRACK_NB_PAIR`: the pair whose Right relation is queried in section 6.

**Run order**: top to bottom. Sections 1-5 are reads plus a dry-run upload plan. Section 6 runs a `SELECT` on the warehouse (this starts a stopped warehouse and costs a little). Section 7 lists job runs. Compute/warehouse start-stop and the real upload are skipped while `DRY_RUN = True`.

**What to look at**: `login` names you and the auth type; the upload plan lists the remote paths (source notebooks lose their `.py`, `.ipynb` imports as Jupyter, git-ignored files are counted as skipped); `run_sql` returns rows for the window.

**Failure modes**
- `ImportError ... dtrack[dbx]` - run `uv sync --extra dbx`.
- `ValueError: profile ... not found` / auth errors - check `~/.databrickscfg` and re-login.
- `PERMISSION_DENIED` on `clusters.get` / `warehouses.get` - you lack CAN_ATTACH_TO / CAN_USE.
- `statement FAILED: TABLE_OR_VIEW_NOT_FOUND` - the Right `table`/`catalog`/`schema` of the pair is wrong or not granted.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                             # pair whose Right side is queried
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "")                              # "" = search for .env upward from the cwd
TO_DATE = None                                                                # None = yesterday
DAYS = 3                                                                      # window for the Right count
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                    # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                  # SQL warehouse id ("" = list them)
CLUSTER_ID = os.environ.get("DTRACK_DBX_CLUSTER_ID", "")                      # cluster id ("" = list them)
JOB_ID = None                                                                 # int to narrow list_runs; None = all jobs
OPEN_BROWSER = False                                                          # True = login() opens the workspace in a browser
UPLOAD_SOURCE = None                                                          # local file/folder; None = the installed dtrack_right package
UPLOAD_TARGET = None                                                          # absolute workspace path; None = /Users/<you>/<name>
OVERWRITE = False                                                             # upload overwrite flag
CLUSTER_ACTION = None                                                         # None | "start" | "stop" (needs DRY_RUN = False)
WAREHOUSE_ACTION = None                                                       # None | "start" | "stop" (needs DRY_RUN = False)
WAIT = False                                                                  # block until start/stop finishes
SQL_LIMIT = 20                                                                # rows run_sql returns
RUNS_LIMIT = 10                                                               # job runs to list
DRY_RUN = True                                                                # False = really upload / start / stop

In [ ]:
import configparser
import json
from datetime import date, timedelta

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from dtrack_left.backends import databricks
from dtrack_left.cli.config import load_config

# project .env (may carry DATABRICKS_* or DTRACK_DBX_* settings)
env_file = ENV_FILE or find_dotenv(usecwd=True)
print("env file:", env_file or "(none found)")
if env_file:
    load_dotenv(env_file, override=False)

# the pair and its window
config = load_config(CONFIG_PATH)
pair = json.loads(json.dumps(config["pairs"][PAIR]))
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()
print(f"pair {PAIR}: right kind={pair['right'].get('kind') or 'table'} window {FROM}..{TO}")
print("DRY_RUN:", DRY_RUN)

## 1. Profiles and the workspace client

`workspace(profile)` returns a `WorkspaceClient` for a `~/.databrickscfg` profile (default `databricks.DEFAULT_PROFILE`). The profile list below is read straight from the config file; tokens are never printed.

In [ ]:
# profiles in ~/.databrickscfg (host and auth type only)
cfg_path = Path(os.environ.get("DATABRICKS_CONFIG_FILE") or Path.home() / ".databrickscfg")
parser = configparser.ConfigParser()
parser.read(cfg_path)
rows = [
    {"profile": name, "host": parser[name].get("host"), "auth_type": parser[name].get("auth_type", "(token/default)")}
    for name in ["DEFAULT", *parser.sections()]
    if parser[name].get("host")
]
print(cfg_path, "| default profile in code:", databricks.DEFAULT_PROFILE)
display(pd.DataFrame(rows))

# the client for the chosen profile
w = databricks.workspace(DBX_PROFILE)
print("host:", w.config.host, "| auth:", w.config.auth_type)

## 2. Log in and confirm who you are (`login`)

`login(w, open_browser=...)` optionally opens the workspace URL, then calls `current_user.me()` and returns host, profile, auth type and user.

In [ ]:
# identity behind the profile
me = databricks.login(w, open_browser=OPEN_BROWSER)
me

## 3. Plan an upload (`upload_workspace`, dry run)

`upload_workspace(w, source, target, dry_run=True)` lists what it would upload without touching the workspace. Inside a git worktree with a `.gitignore`, ignored files are skipped (and counted); `.py` files starting with `# Databricks notebook` become source notebooks without the suffix; `.ipynb` imports as a Jupyter notebook; everything else is uploaded as-is.

In [ ]:
import dtrack_right

# what to upload: your choice, or the installed Right package folder
source_path = Path(UPLOAD_SOURCE) if UPLOAD_SOURCE else Path(dtrack_right.__file__).parent
plan = databricks.upload_workspace(w, source_path, UPLOAD_TARGET, overwrite=OVERWRITE, dry_run=True)
print({key: value for key, value in plan.items() if key != "paths"})
for remote in plan["paths"][:30]:
    print("  ", remote)
if len(plan["paths"]) > 30:
    print(f"   ... {len(plan['paths']) - 30} more")

In [ ]:
# how each local file is classified (JUPYTER / PYTHON source notebook / plain file)
local_files = [source_path] if source_path.is_file() else sorted(p for p in source_path.rglob("*") if p.is_file())
kinds = pd.Series([databricks._workspace_notebook(path) or "file" for path in local_files]).value_counts()
print(kinds.to_string())
notebooks = [p.name for p in local_files if databricks._workspace_notebook(p)]
print("notebooks:", notebooks[:10])

In [ ]:
# a relative target is refused before anything is sent
try:
    databricks.upload_workspace(w, source_path, "Shared/not-absolute", dry_run=True)
except ValueError as exc:
    print("ValueError:", exc)

## 4. Upload for real (DRY_RUN gated) and look at the result

With `DRY_RUN = False` this creates the folders and uploads each file, collecting per-file failures (`exit_code` 1 when any failed). Listing the target afterwards is read-only and runs either way.

In [ ]:
# the real upload
if DRY_RUN:
    print(f"DRY_RUN: would upload {plan['planned']} file(s) to {plan['target']} (overwrite={OVERWRITE})")
else:
    result = databricks.upload_workspace(w, source_path, plan["target"], overwrite=OVERWRITE)
    print({key: value for key, value in result.items() if key != "paths"})
    for failure in result["failed"]:
        print("  FAILED", failure)

# what is at the target now (missing before the first real upload)
try:
    for item in w.workspace.list(plan["target"]):
        print(f"  {databricks._value(item.object_type)!s:10} {item.path}")
except Exception as exc:
    print("list:", type(exc).__name__, str(exc)[:200])

## 5. Compute and SQL warehouse status

`compute_status` reports a cluster's state, node types, EC2 instance ids and uptime since the last (re)start; `warehouse_status` reports state, type, size, clusters, sessions and auto-stop. Start/stop only run when `DRY_RUN = False` and an action is set (`compute_stop` terminates, it never deletes permanently).

In [ ]:
# pick a cluster: the configured one, else show what you can see
if not CLUSTER_ID:
    clusters = [
        {"cluster_id": c.cluster_id, "name": c.cluster_name, "state": databricks._value(c.state)}
        for c in w.clusters.list()
    ]
    display(pd.DataFrame(clusters))
else:
    display(pd.Series(databricks.compute_status(w, CLUSTER_ID)).to_frame("value"))

In [ ]:
# optional start/stop of the cluster
if CLUSTER_ID and CLUSTER_ACTION:
    if DRY_RUN:
        print(f"DRY_RUN: would {CLUSTER_ACTION} cluster {CLUSTER_ID} (wait={WAIT})")
    elif CLUSTER_ACTION == "start":
        print(databricks.compute_start(w, CLUSTER_ID, wait=WAIT))
    elif CLUSTER_ACTION == "stop":
        print(databricks.compute_stop(w, CLUSTER_ID, wait=WAIT))

In [ ]:
# pick a warehouse: the configured one, else show what you can see
if not WAREHOUSE_ID:
    display(pd.DataFrame(
        [{"id": h.id, "name": h.name, "state": databricks._value(h.state), "size": h.cluster_size} for h in w.warehouses.list()]
    ))
else:
    display(pd.Series(databricks.warehouse_status(w, WAREHOUSE_ID)).to_frame("value"))

# optional start/stop of the warehouse
if WAREHOUSE_ID and WAREHOUSE_ACTION:
    if DRY_RUN:
        print(f"DRY_RUN: would {WAREHOUSE_ACTION} warehouse {WAREHOUSE_ID} (wait={WAIT})")
    elif WAREHOUSE_ACTION == "start":
        print(databricks.warehouse_start(w, WAREHOUSE_ID, wait=WAIT))
    elif WAREHOUSE_ACTION == "stop":
        print(databricks.warehouse_stop(w, WAREHOUSE_ID, wait=WAIT))

## 6. Run SQL on the warehouse (`run_sql`)

`run_sql(w, statement, warehouse_id, limit=)` submits a statement, polls while it is PENDING/RUNNING, and returns columns, the first `limit` rows and the total row count; a failed statement raises `RuntimeError` with the warehouse error. The statement is the per-day count the Right side's `partition_counts` runs for this pair and window (read-only).

In [ ]:
from dtrack_right.backends.targets import SparkTarget
from dtrack_right.common.dates import bounds_where, day_expr

# the same per-day count SQL the Right side's partition_counts builds (no Spark session needed for the string)
target = SparkTarget(None, pair["right"])
right_date, right_type = pair["right"]["date_col"], pair["right"].get("date_type")
ref = day_expr(right_date, "spark", right_type)
bounds = bounds_where(right_date, dialect="spark", date_type=right_type, from_date=FROM, to_date=TO)
statement = (
    f"SELECT {ref} AS day, COUNT(*) AS n FROM {target.relation()} "
    f"WHERE {target._where(bounds)} GROUP BY {ref} ORDER BY 1"
)
print(statement)

if not WAREHOUSE_ID:
    print("set WAREHOUSE_ID to run this")
else:
    result = databricks.run_sql(w, statement, WAREHOUSE_ID, limit=SQL_LIMIT)
    print("statement_id:", result["statement_id"], "| total rows:", result["total_rows"])
    display(pd.DataFrame(result["rows"], columns=result["columns"]))

In [ ]:
# a statement that must fail: the warehouse error comes back as RuntimeError
if WAREHOUSE_ID:
    try:
        databricks.run_sql(w, "SELECT * FROM dtrack_nb_no_such_table_09", WAREHOUSE_ID)
    except RuntimeError as exc:
        print("RuntimeError:", str(exc)[:300])

## 7. Job runs (`list_runs`, `run_detail`)

`list_runs` returns the newest runs (one-time submits included) as flat rows; `run_detail` adds the state message and every task with its own state and duration.

In [ ]:
# newest runs, optionally for one job
runs = databricks.list_runs(w, limit=RUNS_LIMIT, job_id=JOB_ID)["runs"]
display(pd.DataFrame(runs))

# the newest run in detail (failed runs carry the explaining message)
if runs:
    detail = databricks.run_detail(w, runs[0]["run_id"])
    print("message:", detail["message"])
    display(pd.DataFrame(detail["tasks"]))

## What to check

- Section 1-2: the profile resolves to the right host and `login` names you.
- Section 3: the plan uploads what you expect, `skipped` counts git-ignored files, notebooks lose their suffix; the relative target is refused.
- Section 4: with `DRY_RUN = False`, `exit_code` is 0 and the listing shows the uploaded tree.
- Section 5: cluster/warehouse state, size and uptime look right; nothing started or stopped while `DRY_RUN = True`.
- Section 6: per-day counts come back for the window; the bad statement raises with `TABLE_OR_VIEW_NOT_FOUND`.
- Section 7: runs list newest first; the detail of a failed run shows why.